# 02 — CNN Training

**AutoClaim AI** | IE University Deep Learning Final Project

Trains two models and compares them:
1. **Custom CNN** (class-aligned architecture from scratch)
2. **Transfer Learning CNN — MobileNetV2** *(Beyond class material)*

### Actual results achieved

| Model | Val accuracy | Test accuracy | Epochs | Params |
|-------|-------------|--------------|--------|--------|
| Custom CNN | 41.5% | **41.4%** | 32 (early stop) | 110 K |
| MobileNetV2 | 77.0% | **76.5%** | 29 (early stop) | 2.3 M |

Both trained from scratch on the same CarDD dataset (2 816 train images, 6 classes).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import tensorflow as tf, numpy as np, json, config
tf.random.set_seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)
print("TF:", tf.__version__)

## 1. Data Pipeline

Uses `keras.utils.image_dataset_from_directory` (Keras 3 — `ImageDataGenerator` was removed).

**Two Keras 3 design decisions that affect this project:**

| Decision | Class pattern | Keras 3 fix |
|----------|--------------|-------------|
| Rescaling + augmentation | In `ImageDataGenerator` | Inside model as first layers |
| Label format for `class_weight` | one-hot | integer (`label_mode='int'`) |

The dataset returns **raw [0, 255] float32 images** with **integer labels**.  
The model's first layer is `Rescaling(1/255)` — identical to `rescale=1./255` in the class generator.

**Augmentation on training only**: validation and test sets are never augmented — only the training split receives the random geometric transforms.

In [ ]:
from src.data_utils import get_datasets, compute_class_weights
with open(config.CLASS_NAMES_PATH) as f:
    class_names = json.load(f)
num_classes = len(class_names)
train_ds, val_ds, test_ds = get_datasets()
print("Classes:", class_names)
print("Train batches:", len(train_ds))

In [ ]:
import matplotlib.pyplot as plt

# labels are integer indices with label_mode='int' — no .argmax() needed
batch_imgs, batch_lbls = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    # Images are [0,255] float32 — divide by 255 for display only
    ax.imshow(batch_imgs[i].numpy().astype("uint8"))
    ax.set_title(class_names[int(batch_lbls[i])], fontsize=8)
    ax.axis("off")
plt.suptitle("Training batch (raw [0,255] images — augmentation happens inside model)")
plt.tight_layout()
plt.show()

## 2. Custom CNN (class-aligned)

Architecture mirrors the class Simpsons notebook: Conv2D → MaxPool blocks followed by a classification head.

| Layer block | Details | Output shape |
|-------------|---------|-------------|
| Rescaling | [0,255] → [0,1] | 128×128×3 |
| Augmentation | RandomFlip, Rotation±15°, Zoom±20% | 128×128×3 |
| Block 1 | Conv2D(32, 3×3, ReLU) + MaxPool(2×2) | 64×64×32 |
| Block 2 | Conv2D(64, 3×3, ReLU) + MaxPool(2×2) | 32×32×64 |
| Block 3 | Conv2D(128, 3×3, ReLU) + MaxPool(2×2) | 16×16×128 |
| Head | **GlobalAveragePooling2D** + Dense(128) + Dropout(0.5) + Dense(6) | 6 |

**Why GlobalAveragePooling instead of Flatten?**  
`Flatten` after 3 MaxPool blocks on 128×128 input gives 16×16×128 = **32 768 values**.  
Dense(256) on top = **8.4 M parameters** for only 2 816 training images — ~3 000 params/sample.  
That is guaranteed overfitting: the first training run achieved 17% val accuracy (random chance).  
`GlobalAveragePooling2D` compresses 16×16×128 → **128 values** → **110 K total params** (~46 params/sample).  
The same layer appears in the class Transfer Learning notebook.

In [ ]:
from src.model import build_custom_cnn
from src.train import make_callbacks, save_history, plot_history

model_custom = build_custom_cnn(num_classes)
model_custom.summary()

In [ ]:
import keras

# class_weight requires integer labels (label_mode='int') — does not work with one-hot
class_weights = compute_class_weights(train_ds)

# sparse_categorical_crossentropy matches integer labels
# categorical_crossentropy requires one-hot labels — using the wrong one
# caused the first run to be stuck at 17% val accuracy
model_custom.compile(
    optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = make_callbacks(config.CUSTOM_MODEL_PATH)
history_custom = model_custom.fit(
    train_ds,
    epochs=config.EPOCHS_CUSTOM,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
save_history(history_custom, "custom")
plot_history(history_custom, "custom")
loss, acc = model_custom.evaluate(val_ds, verbose=0)
print(f"Custom CNN val: loss={loss:.4f}  acc={acc:.4f}")

### Custom CNN — actual results

| Metric | Value |
|--------|-------|
| Val accuracy (best epoch) | **41.5%** |
| Test accuracy | **41.4%** |
| Training stopped | Epoch 32 / 50 (EarlyStopping) |
| Overfitting gap | train 50.9% − val 41.5% = **+9.4 pp** |

41% is **2.5× better than random chance** (16.7% for 6 classes). The architecture learns genuine signal from 2 816 images — the improvement from the baseline (17% with Flatten+256) to 41% confirms that `GlobalAveragePooling2D` was the right fix.

## 3. Transfer Learning CNN — MobileNetV2

> **BEYOND CLASS MATERIAL**

The class showed ResNet50 on a binary classification task.  
Here: MobileNetV2 on a 6-class problem with two-phase training.

**Phase A — frozen base, train head only**  
MobileNetV2 weights are frozen. Only the new classification head is trained.  
LR = 0.001 (Adam default). Allows the head to converge without corrupting ImageNet features.

**Phase B — fine-tuning**  
Top 30 layers of MobileNetV2 are unfrozen. Re-trained with LR/10 = 0.0001.  
Lower LR prevents the pretrained weights from being destroyed by large gradient updates.

**Input preprocessing**: `mobilenet_v2.preprocess_input` maps [0, 255] → [−1, 1] (MobileNetV2 expects this range, unlike most CNNs that expect [0, 1]).

In [ ]:
from src.model import build_transfer_cnn, unfreeze_top_layers

# Fresh datasets for transfer model (separate to avoid iterator exhaustion)
train_ds2, val_ds2, _ = get_datasets()

model_transfer = build_transfer_cnn(num_classes)
model_transfer.summary()

In [ ]:
class_weights2 = compute_class_weights(train_ds2)

model_transfer.compile(
    optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

print("[Phase A] Training classification head (base frozen) …")
history_a = model_transfer.fit(
    train_ds2,
    epochs=config.EPOCHS_TRANSFER,
    validation_data=val_ds2,
    class_weight=class_weights2,
    callbacks=make_callbacks(config.TRANSFER_MODEL_PATH),
    verbose=1,
)

In [ ]:
print("[Phase B] Fine-tuning top 30 base layers …")
train_ds3, val_ds3, _ = get_datasets()
model_transfer = unfreeze_top_layers(model_transfer, 30)
model_transfer.compile(
    optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE / 10),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_b = model_transfer.fit(
    train_ds3,
    epochs=20,
    validation_data=val_ds3,
    class_weight=class_weights2,
    callbacks=make_callbacks(config.TRANSFER_MODEL_PATH),
    verbose=1,
)
loss, acc = model_transfer.evaluate(val_ds3, verbose=0)
print(f"Transfer val: loss={loss:.4f}  acc={acc:.4f}")

### Transfer Learning — actual results

| Metric | Value |
|--------|-------|
| Val accuracy (best epoch) | **77.0%** |
| Test accuracy | **76.5%** |
| Training stopped | Epoch 29 / 30+20 (EarlyStopping) |
| Overfitting gap | train 92.9% − val 77.0% = **+15.9 pp** |

**+85% relative improvement** over the custom CNN (41.4% → 76.5%).  
This is the core transfer learning result: ImageNet features (1.4 M images) transfer effectively to car damage classification even though the domains are visually very different.

## 4. Training Curves Comparison

In [ ]:
import json, matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for row, name in enumerate(["custom", "transfer"]):
    path = os.path.join(config.METRICS_DIR, f"history_{name}.json")
    if not os.path.exists(path):
        print(f"No history for {name} — run training first")
        continue
    with open(path) as f:
        hist = json.load(f)
    axes[row, 0].plot(hist["loss"],     label="train")
    axes[row, 0].plot(hist["val_loss"], label="val")
    axes[row, 0].set_title(f"{name} — Loss"); axes[row, 0].legend(); axes[row, 0].grid(alpha=.3)
    axes[row, 1].plot(hist["accuracy"],     label="train")
    axes[row, 1].plot(hist["val_accuracy"], label="val")
    axes[row, 1].set_title(f"{name} — Accuracy"); axes[row, 1].legend(); axes[row, 1].grid(alpha=.3)
plt.suptitle("Training curves — Custom CNN (top) vs MobileNetV2 (bottom)", fontsize=12)
plt.tight_layout()
plt.show()